## Most Simple

In [96]:
# Import stuff for trees with nodes
from anytree import Node, RenderTree
from anytree.exporter import DotExporter
import re

In [97]:
example_text = "\n\n\
Returns and inequality is so strong. that it yields.\n\n\
Another heuristic for.\n\
\n\n\
Outperform everyone else.\n\n"

patterns = ["\n\n", "\n", "."]

total_len = len(example_text)

In [98]:
def simple(example_text, patterns, verbose=False, save=True):
    # Most naive way to split the text
    start_node = Node("Root", parent=None, lvl=0, text=example_text, pattern="")
    nodes = [start_node]
    queue = [start_node]

    while queue:
        # get the info of the current node
        node = queue.pop(0)    
        text = node.text
        lvl = node.lvl
        matches = text.split(patterns[lvl])

        if verbose:
            print(node)
            print(matches)
            print("_" * 80)

            
        # Take out the last match if it is empty
        if matches[-1] == "":
            matches = matches[:-1]
            last_no_pattern = False
        else:
            last_no_pattern = True

        for m in range(len(matches)):
            new_text = matches[m]
            info = f"{lvl}-{m}: text={new_text} || pattern = "

            if (m == len(matches) - 1) and last_no_pattern:
                info += "no"
                new_node = Node(info, parent=node, lvl=lvl+1, text=new_text, pattern="")
            else:
                info += "yes"
                new_node = Node(info, parent=node, lvl=lvl+1, text=new_text, pattern=patterns[lvl])

            if new_text != "" and (lvl + 1) < len(patterns):
                queue.append(new_node)

            nodes.append(new_node)

    if save:
        # Export the tree to a DOT file and visualize it
        DotExporter(start_node).to_picture("tree_simple_python.png")

In [89]:
simple(example_text, patterns, verbose=True, save=True)

Node('/Root', lvl=0, pattern='', text='\n\nReturns and inequality is so strong. that it yields.\n\nAnother heuristic for.\n\n\nOutperform everyone else.\n\n')
['', 'Returns and inequality is so strong. that it yields.', 'Another heuristic for.', '\nOutperform everyone else.', '']
________________________________________________________________________________
Node('/Root/0-1: text=Returns and inequality is so strong. that it yields. || pattern = yes', lvl=1, pattern='\n\n', text='Returns and inequality is so strong. that it yields.')
['Returns and inequality is so strong. that it yields.']
________________________________________________________________________________
Node('/Root/0-2: text=Another heuristic for. || pattern = yes', lvl=1, pattern='\n\n', text='Another heuristic for.')
['Another heuristic for.']
________________________________________________________________________________
Node('/Root/0-3: text=\nOutperform everyone else. || pattern = yes', lvl=1, pattern='\n\n', text

## Zero Copy

    Changes to inital Rust:

    merging
    tokenization
    tokenoffset
    multipattern at one level

    Changes to final Rust:

    parallelization 
    early termination of nodes (depending on tokens)
    => Maybe only start parallelization with token generation once char length drops < 4000, before it is almost impossible for early cutoff to be sensible and that way tokenization bottleneck can be alleviated by using more threads.

In [109]:
def zerocopy(example_text, patterns, verbose=False, save=True):
    # Most naive way to split the text
    info = f"0-{len(example_text)}: text={example_text} || pattern = 0"
    start_node = Node(info, parent=None, lvl=0, start_idx=0, end_idx=len(example_text), pattern=0)
    nodes = [start_node]
    queue = [start_node]

    while queue:
        
        # get the info of the current node
        node = queue.pop(0)    
        lvl = node.lvl

        pattern = patterns[lvl] if patterns[lvl] != "." else r"\." 
        matches = [m.start() + node.start_idx for m in re.finditer(pattern, example_text[node.start_idx:node.end_idx])]

        # Check if last match is pattern at the end
        if matches and matches[-1] == (node.end_idx - len(patterns[lvl])):
            pass
        else:
            matches += [node.end_idx]

        if verbose:
            print(node)
            print(matches)
            print("_" * 80)

        start_next_idx = node.start_idx
        for m in range(len(matches)):
            end_next_idx = matches[m]
            
            info = f"{start_next_idx}-{end_next_idx}: text={example_text[start_next_idx:end_next_idx]} || pattern = "
            if m == len(matches) - 1:
                info += str(node.end_idx - end_next_idx)
                new_node = Node(info, parent=node, lvl=lvl+1, start_idx=start_next_idx, end_idx=end_next_idx, pattern=node.end_idx - end_next_idx)
            else:
                info += str(len(patterns[lvl]))
                new_node = Node(info, parent=node, lvl=lvl+1, start_idx=start_next_idx, end_idx=end_next_idx, pattern=len(patterns[lvl]))

            if new_node.end_idx - new_node.start_idx > 0 and (lvl + 1) < len(patterns):
                queue.append(new_node)

            nodes.append(new_node)
            start_next_idx = end_next_idx + len(patterns[lvl])
    
    if save:
        # Export the tree to a DOT file and visualize it
        DotExporter(start_node).to_picture("tree_zerocopy_python.png")

In [110]:
zerocopy(example_text, patterns, verbose=True, save=True)

Node('/0-108: text=\n\nReturns and inequality is so strong. that it yields.\n\nAnother heuristic for.\n\n\nOutperform everyone else.\n\n || pattern = 0', end_idx=108, lvl=0, pattern=0, start_idx=0)
[0, 54, 78, 106]
________________________________________________________________________________
Node('/0-108: text=\n\nReturns and inequality is so strong. that it yields.\n\nAnother heuristic for.\n\n\nOutperform everyone else.\n\n || pattern = 0/2-54: text=Returns and inequality is so strong. that it yields. || pattern = 2', end_idx=54, lvl=1, pattern=2, start_idx=2)
[54]
________________________________________________________________________________
Node('/0-108: text=\n\nReturns and inequality is so strong. that it yields.\n\nAnother heuristic for.\n\n\nOutperform everyone else.\n\n || pattern = 0/56-78: text=Another heuristic for. || pattern = 2', end_idx=78, lvl=1, pattern=2, start_idx=56)
[78]
________________________________________________________________________________
Node('/0

## Benchmark

In [105]:
%%timeit
simple(example_text, patterns, verbose=False, save=False)

23.7 µs ± 257 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [106]:
%%timeit
zerocopy(example_text, patterns, verbose=False, save=False)

28.9 µs ± 79.2 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
